# Prepare full SEC MD&A sections (data collection only)

This notebook fetches only the original filings in the project’s fixed text manifest, using the public SEC archive. It extracts 10-K Item 7 and 10-Q Item 2. It fits no models and does not change proxy labels. The cache resumes across Colab sessions. Use a normal CPU runtime for this step; model fitting is in the companion GPU notebook.

SEC requests are sequential and limited to two per second. Enter your real identifying User-Agent when prompted. If the SEC returns 403, the request code stops; do not bypass that denial. The default is 3,000 filings per session so each run can checkpoint to Drive.

In [ ]:
%pip -q install "pandas>=2.2,<3" "numpy>=1.26,<3" "pyarrow>=16,<25" "beautifulsoup4>=4.12,<5"

If Colab requests a runtime restart after installation, restart and run all setup cells again.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ROOT=Path("/content/drive/MyDrive/financial-risk-detector")
assert (ROOT/"data/colab_amendment_proxy.csv.gz").is_file(), "Upload the prepared proxy CSV to Drive/data first."
print("Drive root:",ROOT)

In [ ]:
from pathlib import Path
import sys, importlib, importlib.util
MODULES=Path("/content/mda_section_modules"); MODULES.mkdir(parents=True,exist_ok=True)
SOURCES={'research_v5_core.py': '"""Point-in-time features and fixed study design; no model fitting or networking."""\nimport hashlib\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nTARGET = \'y_amendment_proxy\'\nSEEDS = (42, 143, 244)\nFOLDS = ((2018, 2020), (2020, 2022))\nRATIOS = (\'accruals_to_assets\', \'net_margin\', \'gross_margin\', \'debt_to_assets\',\n          \'cash_to_assets\', \'working_capital_to_assets\', \'receivables_to_revenue\',\n          \'inventory_to_revenue\', \'operating_cashflow_to_assets\', \'revenue_to_assets\',\n          \'liabilities_to_assets\', \'retained_earnings_to_assets\', \'filing_delay_days\')\nEXCLUDED = {\'adsh\', \'cik\', \'period\', \'period_date\', \'filed\', \'filed_date\', \'filed_year\',\n            \'sic\', \'afs\', \'wksi\', \'days_to_amendment\', TARGET, \'feature_schema_version\',\n            \'label_observation_cutoff\'}\n\n\ndef file_hash(path):\n    result = hashlib.sha256()\n    with open(path, \'rb\') as handle:\n        for chunk in iter(lambda: handle.read(2**20), b\'\'):\n            result.update(chunk)\n    return result.hexdigest()\n\n\ndef load_data(path):\n    df = pd.read_csv(path, dtype={\'adsh\':str, \'cik\':str}, low_memory=False)\n    required = EXCLUDED | {\'form\', \'fp\', \'sic2\', \'prior_amendment_count\'}\n    if required - set(df):\n        raise ValueError(f\'Missing columns: {sorted(required-set(df))}\')\n    if not df.feature_schema_version.eq(4).all():\n        raise ValueError(\'Use the prepared schema-v4 CSV, not an earlier export.\')\n    df[\'filed_date\'] = pd.to_datetime(df.filed_date, errors=\'raise\')\n    df[\'period_date\'] = pd.to_datetime(df.period_date, errors=\'raise\')\n    df[\'cik\'] = df.cik.astype(str).str.lstrip(\'0\').replace(\'\', \'0\')\n    if not df.adsh.is_unique or df[TARGET].isna().any() or not set(df[TARGET]) <= {0,1}:\n        raise ValueError(\'Invalid labels or duplicate accessions.\')\n    cutoff = pd.to_datetime(df.label_observation_cutoff)\n    if cutoff.nunique() != 1 or cutoff.iloc[0] < pd.Timestamp(\'2026-09-30\'):\n        raise ValueError(\'Require label observation through 2026-09-30.\')\n    if not ((cutoff-df.filed_date).dt.days >= 365).all():\n        raise ValueError(\'An outcome has less than 365 days of follow-up.\')\n    if not set(df.form) <= {\'10-K\',\'10-Q\'}:\n        raise ValueError(\'Predictors must come from original 10-K/10-Q filings.\')\n    df[\'sic2\'] = df.sic2.fillna(\'unknown\').astype(str)\n    return df.reset_index(drop=True)\n\n\ndef balanced(part, seed=42):\n    pos, neg = part[part[TARGET].eq(1)], part[part[TARGET].eq(0)]\n    if pos.empty:\n        raise ValueError(\'Cohort has zero positives.\')\n    controls = []\n    for (year, form), group in pos.groupby([\'filed_year\',\'form\']):\n        pool = neg[neg.filed_year.eq(year) & neg.form.eq(form)]\n        if len(pool) < len(group):\n            raise ValueError(f\'Insufficient controls: {year}, {form}\')\n        controls.append(pool.sample(len(group), replace=False, random_state=seed))\n    return pd.concat([pos, *controls]).sample(frac=1, random_state=seed)\n\n\ndef date_part(df, start, end):\n    return df[df.filed_date.between(pd.Timestamp(start),pd.Timestamp(end))]\n\n\ndef training_pool(df, fit_year, window=8):\n    return date_part(df, f\'{max(2009,fit_year-window+1)}-01-01\', f\'{fit_year}-12-31\')\n\n\ndef add_mature_history(df):\n    """A prior label is usable only AFTER its complete 365-day outcome window."""\n    result = pd.DataFrame(index=df.index)\n    for group_cols, prefix in [([\'cik\'],\'issuer\'), ([\'cik\',\'form\'],\'issuer_form\')]:\n        count = np.zeros(len(df)); positive = np.zeros(len(df))\n        for _, group in df.groupby(group_cols, sort=False):\n            history = group.sort_values(\'filed_date\')\n            maturity = (history.filed_date + pd.Timedelta(days=365)).to_numpy(\'datetime64[D]\')\n            at = np.searchsorted(maturity, group.filed_date.to_numpy(\'datetime64[D]\'), side=\'left\')\n            prefix_sum = np.r_[0, history[TARGET].to_numpy().cumsum()]\n            count[group.index] = at\n            positive[group.index] = prefix_sum[at]\n        result[f\'{prefix}_mature_count\'] = count\n        result[f\'{prefix}_mature_positive_count\'] = positive\n        # Fixed smoothing constant; never estimated from later outcome labels.\n        result[f\'{prefix}_mature_rate\'] = (positive + 0.4) / (count + 20)\n    return pd.concat([df, result], axis=1)\n\n\ndef add_financial_history(df):\n    columns = [c for c in RATIOS if c in df]\n    # One aggregate per issuer/form/day prevents same-day information leakage.\n    keys = [\'cik\',\'form\',\'filed_date\']\n    days = df.groupby(keys, as_index=False)[columns].mean().sort_values(keys)\n    groups = days.groupby([\'cik\',\'form\'], sort=False)\n    # Compute all ratios together; per-issuer Python callbacks are very slow\n    # across hundreds of thousands of filings on a standard Colab runtime.\n    past = groups[columns].shift()\n    rolling = past.groupby([days.cik,days.form],sort=False).rolling(3,min_periods=1)\n    means = rolling.mean().reset_index(level=[0,1],drop=True)\n    stds = rolling.std().reset_index(level=[0,1],drop=True)\n    derived = []\n    for col in columns:\n        days[f\'past3_mean_{col}\'] = means[col]\n        days[f\'past3_std_{col}\'] = stds[col]\n        derived += [f\'past3_mean_{col}\', f\'past3_std_{col}\']\n    out = df.merge(days[keys+derived], on=keys, how=\'left\', validate=\'many_to_one\')\n    extra = {}\n    for col in columns:\n        extra[f\'deviation_past3_{col}\'] = out[col]-out[f\'past3_mean_{col}\']\n    out = pd.concat([out, pd.DataFrame(extra,index=out.index)], axis=1)\n    # Calendar-year aggregates come solely from the PREVIOUS filing year.\n    for col in columns:\n        stats = df.groupby([\'filed_year\',\'form\',\'sic2\'])[col].agg([\'median\',\'count\']).reset_index()\n        stats[\'filed_year\'] += 1\n        stats.loc[stats[\'count\'] < 20, \'median\'] = np.nan\n        name = f\'prior_year_peer_median_{col}\'\n        stats = stats.rename(columns={\'median\':name})\n        out = out.merge(stats[[\'filed_year\',\'form\',\'sic2\',name]],\n                        on=[\'filed_year\',\'form\',\'sic2\'], how=\'left\', validate=\'many_to_one\')\n        out[f\'peer_deviation_{col}\'] = out[col]-out[name]\n    out[\'loss_with_positive_cashflow\'] = ((out.net_income_to_assets < 0) &\n                                           (out.operating_cashflow_to_assets > 0)).astype(\'int8\')\n    out[\'profit_with_negative_cashflow\'] = ((out.net_income_to_assets > 0) &\n                                            (out.operating_cashflow_to_assets < 0)).astype(\'int8\')\n    out[\'negative_equity\'] = out.xbrl_StockholdersEquity.lt(0).astype(\'int8\')\n    facts = [c for c in df if c.startswith(\'xbrl_\')]\n    out[\'financial_fact_missing_fraction\'] = out[facts].isna().mean(axis=1)\n    return out\n\n\ndef add_event_history(df, events):\n    """Public EDGAR index events, all strictly before the predictor date."""\n    out = pd.DataFrame(index=df.index)\n    events = events.copy()\n    events[\'filed_date\'] = pd.to_datetime(events.filed_date)\n    events[\'cik\'] = events.cik.astype(str).str.lstrip(\'0\').replace(\'\',\'0\')\n    categories = {\'8k\':[\'8-K\',\'8-K/A\'], \'late\':[\'NT 10-K\',\'NT 10-Q\'],\n                  \'10ka\':[\'10-K/A\'], \'10qa\':[\'10-Q/A\']}\n    for name, forms in categories.items():\n        event_groups = {cik:np.sort(g.filed_date.to_numpy(\'datetime64[D]\'))\n                        for cik,g in events[events.form.isin(forms)].groupby(\'cik\')}\n        counts = {n:np.zeros(len(df), dtype=\'int32\') for n in (90,365,730)}\n        gap = np.full(len(df), np.nan)\n        for cik, rows in df.groupby(\'cik\',sort=False):\n            dates = event_groups.get(cik, np.array([],dtype=\'datetime64[D]\'))\n            query = rows.filed_date.to_numpy(\'datetime64[D]\')\n            end = np.searchsorted(dates,query,side=\'left\')\n            for span in counts:\n                counts[span][rows.index] = end-np.searchsorted(dates,query-np.timedelta64(span,\'D\'),side=\'left\')\n            valid = end > 0\n            gap[rows.index[valid]] = (query[valid]-dates[end[valid]-1]).astype(\'timedelta64[D]\').astype(int)\n        for span,values in counts.items():\n            out[f\'sec_{name}_count_{span}d\'] = values\n        out[f\'sec_{name}_days_since\'] = gap\n    return pd.concat([df,out],axis=1)\n\n\ndef threshold_choice(y, probability):\n    """Maximise the worst of precision/recall/F1/accuracy on CALIBRATION only."""\n    y = np.asarray(y, dtype=int); p = np.asarray(probability, dtype=float)\n    if len(y) == 0 or len(np.unique(y)) < 2 or not np.isfinite(p).all():\n        raise ValueError(\'Threshold selection requires finite scores and both classes.\')\n    order = np.argsort(-p, kind=\'stable\'); ys = y[order]; ps = p[order]\n    ends = np.r_[np.flatnonzero(ps[:-1] != ps[1:]),len(ps)-1]\n    tp = ys.cumsum()[ends]; fp = ends+1-tp\n    recall = tp/y.sum(); precision = tp/(ends+1)\n    f1 = 2*precision*recall/np.maximum(precision+recall,1e-12)\n    accuracy = (tp+(1-y).sum()-fp)/len(y)\n    floor = np.minimum.reduce([precision,recall,f1,accuracy])\n    i = int(np.argmax(floor))\n    return {\'threshold\':float(ps[ends[i]]), \'calibration_min_metric\':float(floor[i]),\n            \'calibration_all_metrics_ge_80\':bool(np.max(floor) >= .80),\n            \'calibration_all_metrics_ge_85\':bool(np.max(floor) >= .85)}\n\n\ndef text_manifest(df):\n    parts = []\n    for fit_year in (2018,2020,2021):\n        pool = training_pool(df,fit_year)\n        parts += [balanced(pool,seed) for seed in SEEDS]\n    for start,end in [(\'2020-01-01\',\'2020-12-31\'),(\'2022-01-01\',\'2022-12-31\'),\n                      (\'2023-01-01\',\'2023-09-30\'),(\'2024-10-01\',\'2025-09-30\')]:\n        parts.append(date_part(df,start,end))\n    return pd.concat(parts).drop_duplicates(\'adsh\')[[\'adsh\',\'cik\',\'form\',\'filed_date\']].sort_values(\'adsh\')\n', 'sec_v5_evidence.py': '"""Public SEC evidence: complete quarterly indexes and optional ORIGINAL text.\n\nOnly used for predictor enrichment. Existing proxy outcome labels are retained.\nNo contact identity is stored. One sequential client, at most two requests/sec.\n"""\nimport gzip\nimport hashlib\nimport io\nimport importlib.metadata\nimport json\nimport re\nimport sqlite3\nimport time\nfrom pathlib import Path\nfrom contextlib import closing\nfrom urllib.error import HTTPError, URLError\nfrom urllib.request import Request, urlopen\nfrom zipfile import ZipFile\nimport numpy as np\nimport pandas as pd\n\nFORMS = {\'10-K\',\'10-Q\',\'10-K/A\',\'10-Q/A\',\'8-K\',\'8-K/A\',\'NT 10-K\',\'NT 10-Q\'}\nRISK_PATTERNS = {\n    \'weakness\':r\'\\bmaterial weaknesses?\\b\',\n    \'ineffective\':r\'\\b(?:not effective|ineffective)\\b\',\n    \'deficiency\':r\'\\bsignificant deficienc(?:y|ies)\\b\',\n    \'going_concern\':r\'\\bgoing concern\\b\',\n    \'restatement\':r\'\\brestat(?:ement|ements|ed)\\b\',\n    \'related_party\':r\'\\brelated.part(?:y|ies)\\b\',\n    \'estimate\':r\'\\b(?:critical accounting|accounting estimates?)\\b\',\n    \'late\':r\'\\b(?:late filing|untimely|unable to timely)\\b\',\n    \'auditor_change\':r\'\\b(?:auditor resignation|dismissed.*?auditor)\\b\',\n    \'remediation\':r\'\\bremediat(?:ion|e|ed)\\b\',\n    \'controls\':r\'\\b(?:internal controls?|disclosure controls?)\\b\',\n}\n\n\nclass SecClient:\n    def __init__(self, user_agent):\n        if not re.search(r\'\\S+@\\S+\\.\\S+\', user_agent) or \' \' not in user_agent.strip():\n            raise ValueError(\'SEC_USER_AGENT must contain a real name/organization and contact email.\')\n        self.user_agent = user_agent.strip()\n        self.previous = 0.0\n\n    def get(self, url, limit=80*1024**2, original_form=None):\n        if not url.startswith(\'https://www.sec.gov/Archives/\'):\n            raise ValueError(\'Only public SEC archive URLs are allowed.\')\n        for attempt in range(4):\n            time.sleep(max(0, .5-(time.monotonic()-self.previous)))\n            self.previous = time.monotonic()\n            try:\n                request = Request(url, headers={\'User-Agent\':self.user_agent,\'Accept-Encoding\':\'identity\'})\n                with urlopen(request, timeout=90) as response:\n                    chunks=[]; size=0\n                    while True:\n                        block=response.read(64*1024)\n                        if not block:\n                            break\n                        chunks.append(block); size+=len(block)\n                        if size > limit:\n                            raise ValueError(\'SEC response exceeded the configured size limit.\')\n                        if original_form and b\'</DOCUMENT>\' in block.upper():\n                            joined=b\'\'.join(chunks)\n                            if original_document(joined,original_form) is not None:\n                                return joined\n                    return b\'\'.join(chunks)\n            except HTTPError as exc:\n                if exc.code == 403:\n                    raise RuntimeError(\'SEC denied access (403). Stop requests; use an allowed connection later. No bypass is attempted.\') from exc\n                if exc.code not in (429,500,502,503,504):\n                    raise\n                if attempt == 3:\n                    raise RuntimeError(\'SEC rate/server error persisted. Cached progress is retained; retry later.\') from exc\n                time.sleep(min(60,10*2**attempt))\n            except (URLError,TimeoutError) as exc:\n                if attempt == 3:\n                    raise RuntimeError(\'Network unavailable; cached progress is retained.\') from exc\n                time.sleep(3*2**attempt)\n\n\ndef parse_master(raw):\n    records=[]\n    for line in raw.decode(\'latin-1\').splitlines():\n        fields=line.split(\'|\')\n        if len(fields) != 5 or not fields[0].isdigit() or fields[2] not in FORMS:\n            continue\n        cik,_,form,date,path=fields\n        match=re.fullmatch(r\'edgar/data/(\\d+)/(\\d{10}-\\d{2}-\\d{6})\\.txt\',path)\n        if not match or int(match[1]) != int(cik):\n            continue\n        records.append((str(int(cik)),form,date,match[2],path))\n    if not records:\n        raise ValueError(\'SEC index did not contain expected filing records.\')\n    frame=pd.DataFrame(records,columns=[\'cik\',\'form\',\'filed_date\',\'adsh\',\'archive_path\'])\n    frame[\'filed_date\']=pd.to_datetime(frame.filed_date,errors=\'raise\')\n    return frame.drop_duplicates([\'cik\',\'adsh\'])\n\n\ndef load_sec_history(cache, client=None):\n    cache=Path(cache); cache.mkdir(parents=True,exist_ok=True)\n    parts=[]\n    # Predictor dates end in 2025Q3. Later index events are unnecessary.\n    for year in range(2009,2026):\n        for quarter in range(1,5):\n            if year==2025 and quarter>3:\n                break\n            path=cache/f\'{year}Q{quarter}.parquet\'\n            if not path.exists():\n                if client is None:\n                    raise FileNotFoundError(f\'Run the evidence notebook first: missing {path.name}\')\n                url=f\'https://www.sec.gov/Archives/edgar/full-index/{year}/QTR{quarter}/master.zip\'\n                blob=client.get(url)\n                with ZipFile(io.BytesIO(blob)) as z:\n                    member=next((n for n in z.namelist() if n.lower()==\'master.idx\'),None)\n                    if member is None:\n                        raise ValueError(\'Unexpected SEC index ZIP layout.\')\n                    part=parse_master(z.read(member))\n                temp=path.with_suffix(\'.part.parquet\')\n                part.to_parquet(temp,index=False); temp.replace(path)\n                print(f\'Cached {year}Q{quarter}: {len(part):,} relevant filings\',flush=True)\n            part=pd.read_parquet(path)\n            parts.append(part)\n    return pd.concat(parts,ignore_index=True).drop_duplicates([\'cik\',\'adsh\'])\n\n\ndef original_document(blob, form):\n    if form not in (\'10-K\',\'10-Q\'):\n        raise ValueError(\'Text predictors must be an original 10-K or 10-Q.\')\n    text=blob.decode(\'utf-8\',errors=\'replace\')\n    for doc in re.findall(r\'<DOCUMENT>(.*?)</DOCUMENT>\',text,flags=re.I|re.S):\n        kind=re.search(r\'<TYPE>\\s*([^\\r\\n<]+)\',doc,flags=re.I)\n        body=re.search(r\'<TEXT>(.*?)</TEXT>\',doc,flags=re.I|re.S)\n        if kind and body and kind[1].strip().upper()==form:\n            return body[1]\n    return None\n\n\ndef compact_text(html):\n    from bs4 import BeautifulSoup\n    soup=BeautifulSoup(html,\'html.parser\')\n    for tag in soup.find_all([\'script\',\'style\',\'ix:header\',\'ix:hidden\']):\n        tag.decompose()\n    plain=re.sub(r\'\\s+\',\' \',soup.get_text(\' \',strip=True))\n    words=len(plain.split())\n    if words<200:\n        return \'\', {\'text_available\':0.0,\'text_word_log\':float(np.log1p(words))}\n    features={\'text_available\':1.0,\'text_word_log\':float(np.log1p(words))}\n    # Compact excerpts are screening windows, not certified SEC item sections.\n    excerpts=[plain[:1000]]\n    for name,pattern in RISK_PATTERNS.items():\n        matches=list(re.finditer(pattern,plain,flags=re.I))\n        features[f\'text_{name}_per_1k\']=len(matches)*1000/max(words,1)\n        for match in matches[:2]:\n            excerpts.append(plain[max(0,match.start()-250):match.end()+500])\n    return \'\\n\'.join(dict.fromkeys(excerpts))[:10000],features\n\n\ndef text_db(path):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    db=sqlite3.connect(path)\n    db.execute(\'CREATE TABLE IF NOT EXISTS texts (adsh TEXT PRIMARY KEY, url TEXT, status TEXT, body BLOB, lexical TEXT, sha256 TEXT)\')\n    db.commit()\n    return db\n\n\ndef collect_text(manifest, db_path, client, max_new=3000):\n    db=text_db(db_path)\n    done={r[0] for r in db.execute(\'SELECT adsh FROM texts\')}\n    pending=manifest[~manifest.adsh.isin(done)]\n    print(f\'Text cache: {len(manifest)-len(pending):,}/{len(manifest):,} attempted; \'\n          f\'{len(pending):,} pending. This session limit: {max_new or "all"}.\',flush=True)\n    try:\n        for i,row in enumerate(pending.itertuples(index=False),1):\n            if max_new and i>max_new:\n                break\n            if not re.fullmatch(r\'\\d{10}-\\d{2}-\\d{6}\',row.adsh):\n                raise ValueError(\'Invalid accession in text manifest.\')\n            url=f\'https://www.sec.gov/Archives/edgar/data/{int(row.cik)}/{row.adsh}.txt\'\n            body=\'\'; features={\'text_available\':0.0}; digest=\'\'\n            try:\n                blob=client.get(url,limit=25*1024**2,original_form=row.form)\n                html=original_document(blob,row.form)\n                if html is None:\n                    status=\'original_document_missing\'\n                else:\n                    body,features=compact_text(html)\n                    digest=hashlib.sha256(html.encode()).hexdigest()\n                    status=\'ok\' if body else \'insufficient_text\'\n            except HTTPError as exc:\n                if exc.code != 404:\n                    raise\n                status=\'http_404\'\n            except ValueError as exc:\n                if \'size limit\' not in str(exc):\n                    raise\n                status=\'size_limit\'\n            db.execute(\'INSERT OR REPLACE INTO texts VALUES (?,?,?,?,?,?)\',\n                       (row.adsh,url,status,gzip.compress(body.encode()),json.dumps(features),digest))\n            if i%25==0:\n                db.commit(); print(f\'Text requests completed this session: {i:,}\',flush=True)\n    finally:\n        db.commit(); db.close()\n    return text_coverage(manifest,db_path)\n\n\ndef text_coverage(manifest,db_path):\n    with closing(text_db(db_path)) as db:\n        states=dict(db.execute(\'SELECT adsh,status FROM texts\'))\n    status=manifest.adsh.map(states)\n    report={\'requested\':len(manifest),\'attempted\':int(status.notna().sum()),\n            \'usable\':int(status.eq(\'ok\').sum()),\'pending\':int(status.isna().sum())}\n    print(\'Text coverage:\',report,flush=True)\n    return report\n\n\ndef gpu_text_features(manifest,db_path,cache):\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA required for text embedding.\')\n    coverage=text_coverage(manifest,db_path)\n    if coverage[\'pending\']:\n        raise RuntimeError(\'Text download is incomplete. Resume the evidence notebook, or explicitly set USE_TEXT=False.\')\n    if coverage[\'usable\']/coverage[\'requested\'] < .90:\n        raise RuntimeError(\'Text coverage below 90%; inspect failures before enabling the text experiment.\')\n    from sentence_transformers import SentenceTransformer\n    from huggingface_hub import model_info\n    cache=Path(cache); cache.mkdir(parents=True,exist_ok=True)\n    meta_path=cache/\'encoder.json\'\n    name=\'sentence-transformers/all-MiniLM-L6-v2\'\n    if meta_path.exists():\n        meta=json.loads(meta_path.read_text())\n    else:\n        meta={\'model\':name,\'revision\':model_info(name).sha,\'license\':\'Apache-2.0\',\n              \'source\':f\'https://huggingface.co/{name}\',\'text_extraction_version\':1}\n        meta_path.write_text(json.dumps(meta,indent=2))\n    encoder=SentenceTransformer(meta[\'model\'],revision=meta[\'revision\'],device=\'cuda\',trust_remote_code=False)\n    frames=[]\n    with closing(text_db(db_path)) as db:\n        for start in range(0,len(manifest),1000):\n            ids=manifest.adsh.iloc[start:start+1000].tolist()\n            raw=[db.execute(\'SELECT body,lexical,sha256 FROM texts WHERE adsh=?\',(a,)).fetchone() for a in ids]\n            packages={n:importlib.metadata.version(n) for n in\n                      (\'torch\',\'sentence-transformers\',\'transformers\')}\n            key=hashlib.sha256(json.dumps([ids,[r[2] for r in raw],meta,packages],sort_keys=True).encode()).hexdigest()[:24]\n            path=cache/f\'{key}.parquet\'\n            if path.exists():\n                frames.append(pd.read_parquet(path)); continue\n            pieces=[]; owners=[]; lex=[]\n            for i,(body,features,_) in enumerate(raw):\n                lex.append(json.loads(features)); words=gzip.decompress(body).decode().split()\n                for offset in range(0,min(len(words),8*140),140):\n                    pieces.append(\' \'.join(words[offset:offset+140])); owners.append(i)\n            vectors=np.zeros((len(ids),384),dtype=np.float32); counts=np.zeros(len(ids))\n            if pieces:\n                encoded=encoder.encode(pieces,batch_size=64,show_progress_bar=True,\n                                       convert_to_numpy=True,normalize_embeddings=True)\n                np.add.at(vectors,np.asarray(owners),encoded)\n                np.add.at(counts,np.asarray(owners),1)\n                vectors/=np.maximum(counts[:,None],1)\n            frame=pd.concat([pd.DataFrame({\'adsh\':ids}),pd.DataFrame(lex),\n                             pd.DataFrame(vectors,columns=[f\'text_emb_{j}\' for j in range(384)])],axis=1)\n            temporary=path.with_suffix(\'.part.parquet\')\n            frame.to_parquet(temporary,index=False); temporary.replace(path)\n            frames.append(frame)\n    del encoder\n    torch.cuda.empty_cache()\n    return pd.concat(frames,ignore_index=True).fillna(0)\n\n\nif __name__==\'__main__\':\n    import argparse\n    import getpass\n    import os\n    import research_v5_core as core\n    parser=argparse.ArgumentParser(description=\'Cache public SEC evidence without fitting any model.\')\n    parser.add_argument(\'--root\',default=\'.\')\n    parser.add_argument(\'--text\',action=\'store_true\',help=\'Also download optional original-filing excerpts.\')\n    parser.add_argument(\'--max-new\',type=int,default=3000,help=\'Text requests this session; 0 means all pending.\')\n    args=parser.parse_args()\n    if args.max_new<0:\n        parser.error(\'--max-new must be nonnegative\')\n    root=Path(args.root)\n    client=SecClient(os.environ.get(\'SEC_USER_AGENT\') or getpass.getpass(\'Real SEC User-Agent (name/organization and email): \'))\n    events=load_sec_history(root/\'evidence_v5/indexes\',client)\n    print(f\'Index cache complete: {len(events):,} relevant events.\')\n    if args.text:\n        path=root/\'data/colab_amendment_proxy.csv.gz\'\n        if not path.exists():\n            path=root/\'colab_amendment_proxy.csv.gz\'\n        manifest=core.text_manifest(core.load_data(path))\n        manifest.to_csv(root/\'evidence_v5/text_manifest.csv.gz\',index=False,compression=\'gzip\')\n        collect_text(manifest,root/\'evidence_v5/original_text.sqlite\',client,args.max_new)\n', '19_mda_advanced.py': '"""Collect full MD&A sections from the public SEC archive for the GPU study.\n\nThis module only prepares data. It never fits a model and never changes labels.\nThe source manifest is the existing, fixed text manifest; all outcomes are\nrejoined from the current schema-v4 dataset by accession number.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport json\nimport re\nimport sqlite3\nimport sys\nfrom pathlib import Path\nfrom io import TextIOWrapper\n\nimport pandas as pd\nfrom bs4 import BeautifulSoup\n\nSRC = Path(__file__).resolve().parent\nif str(SRC) not in sys.path:\n    sys.path.insert(0, str(SRC))\n\nimport research_v5_core as core\nimport sec_v5_evidence as evidence\n\nMAX_SECTION_WORDS = 6000\nMIN_SECTION_WORDS = 200\n\n\ndef _item_pattern(form: str, start: bool) -> re.Pattern[str]:\n    if form == \'10-K\':\n        item = r\'7\' if start else r\'(?:7\\s*A|8)\'\n    elif form == \'10-Q\':\n        item = r\'2\' if start else r\'(?:3|4)\'\n    else:\n        raise ValueError(\'Only original 10-K and 10-Q filings have supported MD&A items.\')\n    return re.compile(r\'^\\s*(?:PART\\s+[IVX]+\\s+)?ITEM\\s+\' + item + r\'(?![A-Z0-9])\', re.I)\n\n\ndef extract_mda_section(html: str, form: str, max_words: int = MAX_SECTION_WORDS) -> tuple[str, str, int]:\n    """Extract the longest well-bounded Item 7/Item 2 section.\n\n    SEC filings often repeat section labels in a table of contents. Candidate\n    start/end pairs are scored by body length, which generally selects the real\n    section. Ambiguous or short documents are marked unavailable instead of\n    silently substituting the whole filing.\n    """\n    if form not in {\'10-K\', \'10-Q\'}:\n        raise ValueError(\'Only original 10-K and 10-Q filings are accepted.\')\n    if max_words < MIN_SECTION_WORDS:\n        raise ValueError(f\'max_words must be at least {MIN_SECTION_WORDS}.\')\n    soup = BeautifulSoup(html or \'\', \'html.parser\')\n    for tag in soup.find_all([\'script\', \'style\', \'ix:header\', \'ix:hidden\']):\n        tag.decompose()\n    lines = [re.sub(r\'\\s+\', \' \', line).strip() for line in soup.get_text(\'\\n\').splitlines()]\n    lines = [line for line in lines if line]\n    if not lines:\n        return \'\', \'no_text\', 0\n\n    starts = _item_pattern(form, True)\n    ends = _item_pattern(form, False)\n    start_indices = [i for i, line in enumerate(lines) if starts.search(line)]\n    candidates = []\n    for start in start_indices:\n        end = next((i for i in range(start + 1, len(lines)) if ends.search(lines[i])), None)\n        if end is None:\n            continue\n        body = \'\\n\'.join(lines[start + 1:end]).strip()\n        words = body.split()\n        if len(words) >= MIN_SECTION_WORDS:\n            candidates.append((len(words), body))\n    if not candidates:\n        return \'\', \'no_mda_section\', 0\n\n    word_count, body = max(candidates, key=lambda item: item[0])\n    if word_count > max_words:\n        body = \' \'.join(body.split()[:max_words])\n        return body, \'ok_truncated\', max_words\n    return body, \'ok\', word_count\n\n\ndef build_manifest(root: Path) -> pd.DataFrame:\n    """Return the fixed 111k-style study manifest with current proxy labels."""\n    root = Path(root)\n    proxy_path = root / \'data\' / \'colab_amendment_proxy.csv.gz\'\n    if not proxy_path.is_file():\n        raise FileNotFoundError(f\'Missing prepared proxy dataset: {proxy_path}\')\n    cache_dir = root / \'evidence_v5\' / \'mda_sections\'\n    manifest_path = cache_dir / \'text_manifest.csv.gz\'\n    metadata_path = cache_dir / \'text_manifest.json\'\n    dataset_hash = core.file_hash(proxy_path)\n    if manifest_path.is_file() and metadata_path.is_file():\n        metadata = json.loads(metadata_path.read_text(encoding=\'utf-8\'))\n        if metadata.get(\'proxy_sha256\') == dataset_hash:\n            manifest = pd.read_csv(manifest_path, compression=\'gzip\',\n                                   dtype={\'adsh\':str,\'cik\':str,\'form\':str,\'y_proxy\':\'int8\'})\n            manifest[\'filed_date\'] = pd.to_datetime(manifest.filed_date, errors=\'raise\')\n            return manifest\n    proxy = core.load_data(proxy_path)\n    manifest = core.text_manifest(proxy)\n    labels = proxy[[\'adsh\', \'cik\', \'form\', \'filed_date\', core.TARGET]].copy()\n    manifest = manifest[[\'adsh\']].merge(labels, on=\'adsh\', how=\'left\', validate=\'one_to_one\')\n    if manifest[core.TARGET].isna().any() or manifest.cik.isna().any():\n        raise ValueError(\'A selected accession has no current proxy label or issuer ID.\')\n    manifest = manifest.rename(columns={core.TARGET: \'y_proxy\'})\n    manifest[\'cik\'] = manifest.cik.astype(str).str.lstrip(\'0\').replace(\'\', \'0\')\n    manifest[\'filed_date\'] = pd.to_datetime(manifest.filed_date, errors=\'raise\')\n    if manifest.adsh.duplicated().any():\n        raise ValueError(\'The text manifest must contain unique accessions.\')\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    temporary = manifest_path.with_suffix(\'.part.csv.gz\')\n    manifest.to_csv(temporary, index=False, compression={\'method\':\'gzip\',\'compresslevel\':6,\'mtime\':0})\n    temporary.replace(manifest_path)\n    metadata_path.write_text(json.dumps({\'proxy_sha256\':dataset_hash,\'rows\':len(manifest)},indent=2),\n                             encoding=\'utf-8\')\n    return manifest\n\n\ndef open_section_db(path: Path) -> sqlite3.Connection:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    db = sqlite3.connect(path)\n    db.execute(\'\'\'CREATE TABLE IF NOT EXISTS mda_sections (\n        adsh TEXT PRIMARY KEY, cik TEXT NOT NULL, form TEXT NOT NULL,\n        filed_date TEXT NOT NULL, status TEXT NOT NULL, body BLOB NOT NULL,\n        word_count INTEGER NOT NULL, sha256 TEXT NOT NULL)\'\'\')\n    db.commit()\n    return db\n\n\ndef collect_sections(root: Path, user_agent: str, max_new: int = 3000) -> dict:\n    """Fetch/resume only selected original filings, respecting the SEC client."""\n    if max_new < 0:\n        raise ValueError(\'max_new must be zero (all remaining) or a positive integer.\')\n    root = Path(root)\n    manifest = build_manifest(root)\n    db_path = root / \'evidence_v5\' / \'mda_sections\' / \'sections.sqlite\'\n    db = open_section_db(db_path)\n    done = {row[0] for row in db.execute(\'SELECT adsh FROM mda_sections\')}\n    pending = manifest.loc[~manifest.adsh.isin(done)]\n    client = evidence.SecClient(user_agent)\n    attempted = 0\n    try:\n        for row in pending.itertuples(index=False):\n            if max_new and attempted >= max_new:\n                break\n            attempted += 1\n            adsh = str(row.adsh)\n            cik = str(int(row.cik))\n            accession_path = adsh.replace(\'-\', \'\')\n            url = f\'https://www.sec.gov/Archives/edgar/data/{cik}/{accession_path}/{adsh}.txt\'\n            status = \'http_404\'\n            body = \'\'\n            word_count = 0\n            digest = \'\'\n            try:\n                submission = client.get(url, limit=25 * 1024**2, original_form=row.form)\n                document = evidence.original_document(submission, row.form)\n                if document is None:\n                    status = \'original_document_missing\'\n                else:\n                    body, status, word_count = extract_mda_section(document, row.form)\n                    if body:\n                        digest = hashlib.sha256(body.encode(\'utf-8\')).hexdigest()\n            except Exception as exc:\n                from urllib.error import HTTPError\n                if isinstance(exc, HTTPError) and exc.code == 404:\n                    status = \'http_404\'\n                elif isinstance(exc, ValueError) and \'size limit\' in str(exc):\n                    status = \'size_limit\'\n                else:\n                    raise\n            db.execute(\'INSERT INTO mda_sections VALUES (?,?,?,?,?,?,?,?)\',\n                       (adsh, cik, row.form, pd.Timestamp(row.filed_date).strftime(\'%Y-%m-%d\'),\n                        status, sqlite3.Binary(gzip.compress(body.encode(\'utf-8\'))), word_count, digest))\n            if attempted % 25 == 0:\n                db.commit()\n                print(f\'SEC filing requests this session: {attempted:,}/{len(pending):,}\', flush=True)\n        db.commit()\n    finally:\n        db.commit()\n        db.close()\n    return coverage(root, manifest=manifest)\n\n\ndef coverage(root: Path, manifest: pd.DataFrame | None = None) -> dict:\n    manifest = manifest if manifest is not None else build_manifest(root)\n    db_path = Path(root) / \'evidence_v5\' / \'mda_sections\' / \'sections.sqlite\'\n    if not db_path.is_file():\n        return {\'requested\': len(manifest), \'attempted\': 0, \'usable\': 0, \'pending\': len(manifest)}\n    with open_section_db(db_path) as db:\n        statuses = dict(db.execute(\'SELECT adsh,status FROM mda_sections\'))\n    status = manifest.adsh.map(statuses)\n    return {\'requested\': len(manifest), \'attempted\': int(status.notna().sum()),\n            \'usable\': int(status.isin([\'ok\', \'ok_truncated\']).sum()),\n            \'pending\': int(status.isna().sum()),\n            \'status_counts\': status.fillna(\'pending\').value_counts().sort_index().to_dict()}\n\n\ndef export_dataset(root: Path, output_path: Path | None = None) -> Path:\n    """Export all manifest rows, including explicit missing-text statuses."""\n    root = Path(root)\n    manifest = build_manifest(root)\n    db_path = root / \'evidence_v5\' / \'mda_sections\' / \'sections.sqlite\'\n    if not db_path.is_file():\n        raise FileNotFoundError(f\'No SEC text cache exists yet: {db_path}\')\n    with open_section_db(db_path) as db:\n        statuses = dict(db.execute(\'SELECT adsh,status FROM mda_sections\'))\n        missing = int((~manifest.adsh.isin(statuses)).sum())\n        if missing:\n            raise RuntimeError(f\'{missing:,} filings remain pending; resume collection before export.\')\n        rows = db.execute(\'SELECT adsh,status,body,word_count,sha256 FROM mda_sections\')\n        cached = {adsh: (status, body, words, digest)\n                  for adsh, status, body, words, digest in rows}\n    output_path = output_path or root / \'data\' / \'colab_mda_text_expanded.csv.gz\'\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    with output_path.open(\'wb\') as raw:\n        with gzip.GzipFile(fileobj=raw, mode=\'wb\', compresslevel=6, mtime=0) as compressed:\n            with TextIOWrapper(compressed, encoding=\'utf-8\', newline=\'\') as text_file:\n                writer = csv.writer(text_file)\n                writer.writerow([\'adsh\', \'cik\', \'form\', \'filed_date\', \'y_proxy\', \'mda_text\',\n                                 \'text_status\', \'mda_word_count\', \'text_sha256\'])\n                for row in manifest.itertuples(index=False):\n                    status, body, words, digest = cached[row.adsh]\n                    section = gzip.decompress(body).decode(\'utf-8\') if body else \'\'\n                    writer.writerow([row.adsh, row.cik, row.form,\n                                     pd.Timestamp(row.filed_date).strftime(\'%Y-%m-%d\'),\n                                     int(row.y_proxy), section, status, int(words), digest])\n    return output_path\n\n\ndef prepare(root: Path, user_agent: str, max_new: int = 3000) -> tuple[dict, Path | None]:\n    report = collect_sections(root, user_agent, max_new=max_new)\n    output = None\n    if report[\'pending\'] == 0:\n        output = export_dataset(root)\n        report[\'dataset\'] = str(output)\n        report[\'dataset_bytes\'] = output.stat().st_size\n    return report, output\n\n'}
for name,source in SOURCES.items(): (MODULES/name).write_text(source,encoding="utf-8")
if str(MODULES) not in sys.path: sys.path.insert(0,str(MODULES))
import research_v5_core as core
importlib.reload(core)
import sec_v5_evidence as evidence
importlib.reload(evidence)
print("Embedded research code loaded.")

In [ ]:
spec=importlib.util.spec_from_file_location("mda_advanced",MODULES/"19_mda_advanced.py")
prepare=importlib.util.module_from_spec(spec); spec.loader.exec_module(prepare)
import getpass, json
USER_AGENT=getpass.getpass("SEC User-Agent (real name/organization and contact email): ").strip()
MAX_NEW_PER_SESSION=3000  # Set to 0 only if you intend to fetch every remaining filing this session.
report=prepare.collect_sections(ROOT,USER_AGENT,max_new=MAX_NEW_PER_SESSION)
if report["pending"]==0:
    dataset=prepare.export_dataset(ROOT)
    report["dataset"]=str(dataset); report["dataset_bytes"]=dataset.stat().st_size
print(json.dumps(report,indent=2))
if report["pending"]:
    print("Progress is saved in Drive. Rerun this cell in a later session until pending is 0.")

## Resume and finish

When `pending` is nonzero, rerun the last cell in a later Colab session. Completed rows and manifest fingerprints are cached under `evidence_v5/mda_sections/`. Once the manifest is complete, the notebook writes `data/colab_mda_text_expanded.csv.gz`. Missing or unparseable sections remain as rows with an explicit status and empty text; they are not relabeled or silently removed.